[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Soliman2020/python_48h_notebooks/blob/master/Session_05_Expansion_Lambda_Examples.ipynb)

# Session 05 — Extra examples: lambda functions

Companion material for the **Lambda functions** part of *Session 05 — Functions, Reuse & Modules*.
It starts from the one example in the notebook and adds the cases you meet in real code.

**How to work:** as in the notebook, read the explanation, **predict** the output, run the cell, then
change one value and run it again. Every output below was produced by actually running the code.

> **The idea in one line:** `lambda` builds a small **anonymous** function — one expression, no
> `def`, no `return`, and the value of the expression is returned automatically.

> **When to use it:** when you pass a short rule straight to another function, for example
> `sorted(records, key=lambda record: record['hours'])`. If you need a name, several steps, a
> docstring or an `if`/`elif` chain, use `def` instead.

## Concept 1 — The same rule, two ways

This is the notebook example, with the two forms side by side. Both produce the same result.

In [ ]:
# A named function
def tax_def(amount):
    return amount * 0.15

# The same rule as a lambda: one expression, the result is returned automatically
tax_lambda = lambda amount: amount * 0.15

print(tax_def(200))
print(tax_lambda(200))

**Stop and think:** `tax_def` uses `return`; `tax_lambda` does not. Where does the value
come from? *The value of the single expression is the return value — a lambda always returns it.*

### A lambda does not need a name

Because it is anonymous, you can call it immediately by wrapping it in brackets.

In [ ]:
print((lambda x: x * 2)(5))
print((lambda price, rate: price * (1 - rate))(200, 0.25))

You will rarely write this, but it makes the syntax clear: `lambda x: x * 2` is a value,
just like `5` or `'text'`, and the brackets on the right call it immediately.

### What a lambda cannot do

A lambda body is **one expression**. A statement such as an assignment is not allowed.

In [ ]:
double = lambda x: y = x * 2

The fix is to use one expression, or to move to a `def` when you need several steps:

In [ ]:
double = lambda x: x * 2
print(double(7))

# Several steps, a docstring and a decision are all easier in a def
def double_and_report(value):
    """Return the doubled value and print a short label."""
    doubled = value * 2
    print('doubled:', doubled)
    return doubled

print(double_and_report(7))

## Concept 2 — The arguments you already know

Everything you learned about `def` parameters works in a lambda: defaults, `*args` and keyword
arguments.

In [ ]:
# Default value: used when the caller does not supply one
discount = lambda price, rate=0.10: price * (1 - rate)
print(discount(200))        # default rate
print(discount(200, 0.25))  # caller overrides it

In [ ]:
# *args collects any number of values
total = lambda *values: sum(values)
print(total(3, 5, 2))
print(total(*[10, 20]))     # unpacking a list works too

In [ ]:
# Keyword arguments work as usual
bmi = lambda weight_kg, height_m: weight_kg / height_m ** 2
print(round(bmi(70, 1.75), 2))
print(round(bmi(height_m=1.75, weight_kg=70), 2))

**Watch for:** a lambda cannot carry a docstring or type hints. When you need
`help(my_function)` to explain itself — as in the `greet` example in the notebook — use `def`.

## Concept 3 — Returning a decision

A conditional expression (`A if condition else B`) is a single expression, so it fits inside a
lambda.

In [ ]:
label = lambda score: 'pass' if score >= 70 else 'review'
print(label(76), label(64))

In [ ]:
# The same three-level rule from Session 02, as a lambda
grade = lambda score: 'excellent' if score >= 85 else 'pass' if score >= 70 else 'review'
print(grade(90), grade(76), grade(50))

It works, but compare it with the `if`/`elif`/`else` version from Session 02. The chain of
`if ... else ... if ... else` is hard to read and easy to get wrong. **This is the point where a
`def` is the better choice:**

In [ ]:
def grade_of(score):
    if score >= 85:
        return 'excellent'
    if score >= 70:
        return 'pass'
    return 'review'

print(grade_of(90), grade_of(76), grade_of(50))

## Concept 4 — The everyday use: sorting and picking

This is where you will meet lambda most often: a short `key` rule that tells `sorted`, `min` or
`max` **which part** of each item to compare.

In [ ]:
sites = ['alpha', 'Beta', 'gamma']
print(sorted(sites))                              # capital B sorts before lowercase letters
print(sorted(sites, key=lambda name: name.lower()))  # compare as if all lowercase

### Sorting records by one field

The key function receives one item and returns the value to compare. Here each item is a dictionary,
so the rule is simply "use the `hours_open` field".

In [ ]:
incidents = [
    {'id': 'T-01', 'hours_open': 30, 'priority': 'high'},
    {'id': 'T-02', 'hours_open': 4,  'priority': 'low'},
    {'id': 'T-03', 'hours_open': 12, 'priority': 'high'},
]

worst_first = sorted(incidents, key=lambda record: record['hours_open'], reverse=True)
for record in worst_first:
    print(record['id'], record['hours_open'])

**Compare with Session 04:** the same job with the loop pattern from that session.

In [ ]:
incidents = [
    {'id': 'T-01', 'hours_open': 30, 'priority': 'high'},
    {'id': 'T-02', 'hours_open': 4,  'priority': 'low'},
    {'id': 'T-03', 'hours_open': 12, 'priority': 'high'},
]

# Session 04 style: a loop with a manual comparison
worst = incidents[0]
for record in incidents:
    if record['hours_open'] > worst['hours_open']:
        worst = record
print('Longest open:', worst['id'], worst['hours_open'])

# The same answer, expressed as a rule
worst = max(incidents, key=lambda record: record['hours_open'])
print('Longest open:', worst['id'], worst['hours_open'])

In [ ]:
readings = [
    {'sensor': 'S1', 'value': 22.5},
    {'sensor': 'S2', 'value': 31.0},
    {'sensor': 'S3', 'value': 19.4},
]

highest = max(readings, key=lambda item: item['value'])
lowest = min(readings, key=lambda item: item['value'])
print('Highest:', highest['sensor'], highest['value'])
print('Lowest :', lowest['sensor'], lowest['value'])

### Two rules at once

The key function may return a **tuple**. Python compares the first value, then the second one when
the first is equal. This sorts by priority first, and inside the same priority by the longest
open time.

In [ ]:
rank = {'high': 0, 'medium': 1, 'low': 2}

cases = [
    {'id': 'C1', 'priority': 'low',  'hours': 10},
    {'id': 'C2', 'priority': 'high', 'hours': 2},
    {'id': 'C3', 'priority': 'high', 'hours': 9},
]

# rank first (0 = most urgent), then longest first (minus hours)
ordered = sorted(cases, key=lambda case: (rank[case['priority']], -case['hours']))
for case in ordered:
    print(case['id'], case['priority'], case['hours'])

## Concept 5 — map, filter and reduce

These built-ins take a function as their argument, which is exactly what a lambda is good at.
`map` transforms every item, `filter` keeps the items that pass a test.

In [ ]:
minutes = [90, 45, 15]
hours = list(map(lambda m: round(m / 60, 2), minutes))
print(hours)

In [ ]:
temperatures = [27, -3, 32, None, 36, 0]
valid = list(filter(lambda t: t is not None and t > 0, temperatures))
print(valid)

**Stop and think:** both cells create a **new list** and leave the original untouched.
What would you change to keep 0 as valid?

In [ ]:
readings = [-1.0, 8.3, 12.5, 0.0]
print(list(filter(lambda v: v > 0, readings)))

# The same result as a comprehension (Session 04) - usually easier to read
print([v for v in readings if v > 0])

`reduce` folds a list into one value. It lives in `functools`, and for a simple sum
`sum()` is the better tool — you will still see `reduce` in other people's code.

In [ ]:
from functools import reduce

hours = [4, 7, 3]
print(reduce(lambda a, b: a + b, hours))
print(sum(hours))

# reduce is worth knowing for rules that are not addition
print(reduce(lambda a, b: a if a > b else b, hours))

## Concept 6 — Choosing behaviour from a dictionary

A dictionary of lambdas replaces a long `if`/`elif` chain: the status selects the rule.

In [ ]:
actions = {
    'down': lambda device: f'{device}: investigate',
    'slow': lambda device: f'{device}: monitor',
    'up':   lambda device: f'{device}: archive',
}

print(actions['down']('R01'))
print(actions['slow']('R02'))

# .get with a fallback stops a KeyError for an unknown status
fallback = lambda device: f'{device}: unknown status'
print(actions.get('weird', fallback)('R09'))

## Concept 7 — Where lambda bites you

### 2. Naming a lambda hides the name

Assigning a lambda to a name works, but a traceback will only say `<lambda>`. PEP 8 therefore prefers
`def` when the function has a name.

In [ ]:
def divide_def(a, b):
    return a / b

divide_lambda = lambda a, b: a / b

print(divide_def.__name__)         # a real name: easier to read in an error message
print(divide_lambda.__name__)      # only '<lambda>'

### 3. A lambda is not automatically better

Reach for a lambda when the rule is short and used once, as the `key` for `sorted`, `min` or `max`.
For anything with a name, several steps, a docstring or a long condition, `def` is clearer.

**Watch for:** writing `lambda` where a plain expression would do. `sorted(records, key=lambda r:
r['hours'])` is right; `lambda` around a whole block of logic is not.

## Applied mini-tasks

Short tasks in the cross-sector style of the notebook: telecom incidents, audit cases, laboratory
readings and energy meters. Each one is a `key` or `map`/`filter` rule of two or three lines.

In [ ]:
# Telecom: label every incident, then sort the escalated ones first
incidents = [
    {'id': 'T-01', 'status': 'down', 'hours_open': 30},
    {'id': 'T-02', 'status': 'up',   'hours_open': 4},
    {'id': 'T-03', 'status': 'down', 'hours_open': 12},
]

label = lambda record: 'ESCALATE' if record['status'] == 'down' and record['hours_open'] > 24 else 'normal'
for record in sorted(incidents, key=lambda r: r['hours_open'], reverse=True):
    print(record['id'], label(record))

**Stop and think:** T-03 is `down` for 12 hours. Why is it not escalated? *Because the rule
needs both conditions: `down` **and** more than 24 hours open.*

In [ ]:
# Audit: order cases by priority, then by the largest amount
priority_rank = {'high': 0, 'medium': 1, 'low': 2}
cases = [
    {'case_id': 'C-01', 'priority': 'medium', 'amount': 5000},
    {'case_id': 'C-02', 'priority': 'high',   'amount': 1200},
    {'case_id': 'C-03', 'priority': 'high',   'amount': 9000},
]
for case in sorted(cases, key=lambda c: (priority_rank[c['priority']], -c['amount'])):
    print(case['case_id'], case['priority'], case['amount'])

In [ ]:
# Laboratory: drop invalid readings, then convert to the report format
raw = [12.5, -1.0, None, 8.3, 0.0, 15.7]
clean = filter(lambda v: v is not None and v > 0, raw)
report = list(map(lambda v: f'{v:.1f} C', clean))
print(report)

In [ ]:
# Energy: total consumption from the meters above a threshold
meters = [
    {'meter': 'M1', 'kwh': 85},
    {'meter': 'M2', 'kwh': 12},
    {'meter': 'M3', 'kwh': 63},
]
big = filter(lambda m: m['kwh'] >= 50, meters)
print(sum(map(lambda m: m['kwh'], big)))

# In one line
print(sum(m['kwh'] for m in meters if m['kwh'] >= 50))

## Practice lab — 4 exercises

**How to work:** predict first, write your own version in the cell, then open the solution to
compare. The solution is one way to do it, not the only way.

### Exercise 1 — Foundation · `S05-L01`

Sort `['delta', 'Alpha', 'charlie', 'Bravo']` so that capital letters do not come first.

In [ ]:
# TODO

<details><summary>Click here for the solution</summary>

```python
names = ['delta', 'Alpha', 'charlie', 'Bravo']
print(sorted(names, key=lambda name: name.lower()))
```

</details>

### Exercise 2 — Foundation · `S05-L02`

Use `filter` to keep the readings above 10 from `[5, 12, 3, 19, 10]`. Then write the same result as a
list comprehension.

In [ ]:
# TODO

<details><summary>Click here for the solution</summary>

```python
readings = [5, 12, 3, 19, 10]
print(list(filter(lambda v: v > 10, readings)))
print([v for v in readings if v > 10])
```

</details>

### Exercise 3 — Applied · `S05-L03`

Given the incident records below, print them ordered by the **longest open time first**, and show
`id`, `status` and `hours_open` for each one.

In [ ]:
# TODO

<details><summary>Click here for the solution</summary>

```python
incidents = [
    {'id': 'T-01', 'status': 'down', 'hours_open': 30},
    {'id': 'T-02', 'status': 'up',   'hours_open': 4},
    {'id': 'T-03', 'status': 'down', 'hours_open': 12},
]

for record in sorted(incidents, key=lambda r: r['hours_open'], reverse=True):
    print(record['id'], record['status'], record['hours_open'])
```

</details>

### Exercise 4 — Challenge · `S05-L04`

For the audit cases in Concept 4, print the **most urgent case** using `max` and a key rule, where
`high` is most urgent, then `medium`, then `low`. When two cases share a priority, the one with the
larger amount wins.

In [ ]:
# TODO

<details><summary>Click here for the solution</summary>

```python
rank = {'high': 0, 'medium': 1, 'low': 2}
cases = [
    {'case_id': 'C-01', 'priority': 'medium', 'amount': 5000},
    {'case_id': 'C-02', 'priority': 'high',   'amount': 1200},
    {'case_id': 'C-03', 'priority': 'high',   'amount': 9000},
]

# smallest rank = most urgent; for max, invert both values
most_urgent = max(cases, key=lambda c: (-rank[c['priority']], c['amount']))
print(most_urgent['case_id'], most_urgent['priority'], most_urgent['amount'])
```

</details>

## Exit quiz — answer from memory

1. What does a lambda return, when there is no `return` keyword?
2. How many expressions can the body of a lambda contain?
3. Which built-in would you pass a lambda to in order to sort records by one field?
4. Why does PEP 8 prefer `def` when the function has a name?


<details><summary>Check your answers</summary>

1. The value of its single expression — a lambda returns it automatically.
2. Exactly one. An assignment or any other statement is a `SyntaxError`.
3. `sorted(records, key=lambda record: record['hours'])` — also `min` and `max`.
4. Because a named lambda is usually less readable, and its `__name__` is only `<lambda>`, which
   makes error messages harder to follow. Use `def` and keep lambdas for short, one-off rules.

</details>

## References

- [Lambda expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions) — Python tutorial
- [Sorting HOW TO](https://docs.python.org/3/howto/sorting.html) — `key` functions in depth
- [PEP 8 — Programming Recommendations](https://peps.python.org/pep-0008/#programming-recommendations) — why `def` beats a named lambda
- [Built-in functions](https://docs.python.org/3/library/functions.html) — `map`, `filter`, `sorted`, `min`, `max`
- [functools.reduce](https://docs.python.org/3/library/functools.html#functools.reduce)